# ChurnIQ — Telecom Customer Churn & Revenue Risk Analyzer

## Phase 8 — Revenue Risk Engine & Customer Prioritization

### Objective

Predicting churn alone is not sufficient for business decision-making.

This phase introduces a Revenue Risk Engine that combines:

- Churn Probability
- Customer Revenue

to estimate:

- Revenue at Risk
- Customer Priority Level
- Retention Focus Areas

---

### Business Question

If multiple customers are likely to churn:

Which customers should be retained first?

---

### Champion Model

XGBoost

Threshold: 0.40

Cross-Validated ROC-AUC: 0.680

Recall: 82.66%

F1 Score: 0.495

In [2]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    "../data/processed/feature_engineered.csv"
)

print(df.shape)

(51047, 67)


C:\Users\Akshit\AppData\Local\Temp\ipykernel_10884\1420387941.py:5: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [4]:
# ==========================================
# TARGET ENCODING
# ==========================================

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [5]:
# ==========================================
# FINAL FEATURE SET
# ==========================================

columns_to_drop = [
    "CustomerID",
    "ServiceArea",
    "MarketCode",
    "AreaCode",
    "CreditRating",
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

model_df = df.drop(
    columns=columns_to_drop
)

print(model_df.shape)

(51047, 59)


In [6]:
# ==========================================
# FEATURES & TARGET
# ==========================================

X = model_df.drop(columns=["Churn"])
y = model_df["Churn"]

print(X.shape)
print(y.shape)

(51047, 58)
(51047,)


In [7]:
# ==========================================
# FEATURE TYPES
# ==========================================

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))

Numerical: 38
Categorical: 20


In [8]:
# ==========================================
# CLASS IMBALANCE
# ==========================================

negative_class = (y == 0).sum()
positive_class = (y == 1).sum()

scale_pos_weight = (
    negative_class / positive_class
)

print(round(scale_pos_weight, 2))

2.47


In [9]:
# ==========================================
# PREPROCESSOR
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [10]:
# ==========================================
# TRAIN CHAMPION MODEL
# ==========================================

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

xgb_pipeline.fit(X, y)

print("Champion model trained.")

Champion model trained.


In [11]:
# ==========================================
# CHURN PROBABILITIES
# ==========================================

churn_probabilities = xgb_pipeline.predict_proba(X)[:, 1]

print("Probabilities generated.")

Probabilities generated.


## Step 1: Create Revenue Risk Score

A churn prediction alone does not indicate business impact.

To prioritize retention efforts, a Revenue Risk Score is created:

Revenue Risk Score = Churn Probability × Monthly Revenue

Customers with higher scores represent greater financial risk to the company.

In [12]:
# ==========================================
# REVENUE RISK DATAFRAME
# ==========================================

risk_df = pd.DataFrame()

risk_df["CustomerID"] = df["CustomerID"]

risk_df["MonthlyRevenue"] = df["MonthlyRevenue"]

risk_df["ChurnProbability"] = churn_probabilities

risk_df.head()

,CustomerID,MonthlyRevenue,ChurnProbability
0,3000002,24.00,0.690378
1,3000010,16.99,0.672506
2,3000014,38.00,0.624575
3,3000022,82.28,0.271952
4,3000026,17.14,0.803609


In [13]:
# ==========================================
# REVENUE RISK SCORE
# ==========================================

risk_df["RevenueRiskScore"] = (
    risk_df["MonthlyRevenue"]
    * risk_df["ChurnProbability"]
)

risk_df.head()

,CustomerID,MonthlyRevenue,ChurnProbability,RevenueRiskScore
0,3000002,24.00,0.690378,16.569074
1,3000010,16.99,0.672506,11.425869
2,3000014,38.00,0.624575,23.733864
3,3000022,82.28,0.271952,22.376199
4,3000026,17.14,0.803609,13.773858


## Step 2: Rank Customers by Revenue Risk

Customers with higher revenue risk scores should be prioritized for retention interventions.

In [14]:
# ==========================================
# SORT BY RISK
# ==========================================

risk_df = risk_df.sort_values(
    by="RevenueRiskScore",
    ascending=False
)

risk_df.head(20)

,CustomerID,MonthlyRevenue,ChurnProbability,RevenueRiskScore
24038,3189262,1223.38,0.563428,689.286820
7618,3059858,855.50,0.645068,551.855308
38290,3305434,861.11,0.627066,539.972869
31127,3247882,758.17,0.669646,507.705572
24874,3196158,575.75,0.773669,445.440135
23481,3184774,926.08,0.478221,442.870572
37456,3298662,523.20,0.785112,410.770486
37663,3300310,672.01,0.581583,390.829527
47654,3375582,847.82,0.451721,382.978008
24711,3194730,576.86,0.598706,345.369375


In [15]:
# ==========================================
# TOP 10 REVENUE RISK CUSTOMERS
# ==========================================

risk_df[
    [
        "CustomerID",
        "MonthlyRevenue",
        "ChurnProbability",
        "RevenueRiskScore"
    ]
].head(10)

,CustomerID,MonthlyRevenue,ChurnProbability,RevenueRiskScore
24038,3189262,1223.38,0.563428,689.286820
7618,3059858,855.50,0.645068,551.855308
38290,3305434,861.11,0.627066,539.972869
31127,3247882,758.17,0.669646,507.705572
24874,3196158,575.75,0.773669,445.440135
23481,3184774,926.08,0.478221,442.870572
37456,3298662,523.20,0.785112,410.770486
37663,3300310,672.01,0.581583,390.829527
47654,3375582,847.82,0.451721,382.978008
24711,3194730,576.86,0.598706,345.369375


In [16]:
# ==========================================
# SUMMARY STATISTICS
# ==========================================

risk_df[
    [
        "MonthlyRevenue",
        "ChurnProbability",
        "RevenueRiskScore"
    ]
].describe()

,MonthlyRevenue,ChurnProbability,RevenueRiskScore
count,51047.000000,51047.000000,51047.000000
mean,58.802788,0.467198,27.030329
std,44.442964,0.156729,22.821935
min,-6.170000,0.056093,-4.325924
25%,33.660000,0.351423,14.194011
50%,48.460000,0.477154,21.251607
75%,70.960000,0.579418,32.946433
max,1223.380000,0.930895,689.286820


In [17]:
(df["MonthlyRevenue"] < 0).sum()

np.int64(3)

In [18]:
df[df["MonthlyRevenue"] < 0][
    ["CustomerID", "MonthlyRevenue"]
].head(20)

,CustomerID,MonthlyRevenue
26596,3210322,-2.52
33352,3265738,-5.86
48038,3378298,-6.17


In [19]:
risk_df["MonthlyRevenue_Clean"] = (
    risk_df["MonthlyRevenue"]
    .clip(lower=0)
)

In [20]:
risk_df["RevenueRiskScore"] = (
    risk_df["MonthlyRevenue_Clean"]
    *
    risk_df["ChurnProbability"]
)

In [21]:
risk_df[
    [
        "MonthlyRevenue_Clean",
        "RevenueRiskScore"
    ]
].describe()

,MonthlyRevenue_Clean,RevenueRiskScore
count,51047.000000,51047.000000
mean,58.803073,27.030482
std,44.442570,22.821744
min,0.000000,0.000000
25%,33.660000,14.194011
50%,48.460000,21.251607
75%,70.960000,32.946433
max,1223.380000,689.286820


In [22]:
risk_df["MonthlyRevenue_Clean"] = (
    risk_df["MonthlyRevenue"]
    .clip(lower=0)
)

risk_df["RevenueRiskScore"] = (
    risk_df["MonthlyRevenue_Clean"]
    *
    risk_df["ChurnProbability"]
)

In [23]:
risk_df[
    [
        "MonthlyRevenue_Clean",
        "RevenueRiskScore"
    ]
].describe()

,MonthlyRevenue_Clean,RevenueRiskScore
count,51047.000000,51047.000000
mean,58.803073,27.030482
std,44.442570,22.821744
min,0.000000,0.000000
25%,33.660000,14.194011
50%,48.460000,21.251607
75%,70.960000,32.946433
max,1223.380000,689.286820


## Step 3: Customer Priority Segmentation

Customers are segmented based on Revenue Risk Score.

Priority levels help retention teams focus on customers with the highest potential financial impact.

Segments:

- High Priority
- Medium Priority
- Low Priority

In [24]:
# ==========================================
# PRIORITY SEGMENTS
# ==========================================

risk_df["PrioritySegment"] = pd.qcut(
    risk_df["RevenueRiskScore"],
    q=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

risk_df.head()

,CustomerID,MonthlyRevenue,ChurnProbability,RevenueRiskScore,MonthlyRevenue_Clean,PrioritySegment
24038,3189262,1223.38,0.563428,689.286820,1223.38,High
7618,3059858,855.50,0.645068,551.855308,855.50,High
38290,3305434,861.11,0.627066,539.972869,861.11,High
31127,3247882,758.17,0.669646,507.705572,758.17,High
24874,3196158,575.75,0.773669,445.440135,575.75,High


In [25]:
# ==========================================
# SEGMENT DISTRIBUTION
# ==========================================

risk_df["PrioritySegment"].value_counts()

PrioritySegment
Low       17016
High      17016
Medium    17015
Name: count, dtype: int64

In [26]:
# ==========================================
# SEGMENT PERCENTAGES
# ==========================================

round(
    risk_df["PrioritySegment"]
    .value_counts(normalize=True)
    * 100,
    2
)

PrioritySegment
Low       33.33
High      33.33
Medium    33.33
Name: proportion, dtype: float64

## Step 4: Revenue At Risk Estimation

Estimate the total revenue currently at risk based on:

Revenue Risk Score = Churn Probability × Monthly Revenue

In [27]:
# ==========================================
# TOTAL REVENUE AT RISK
# ==========================================

total_revenue_at_risk = (
    risk_df["RevenueRiskScore"]
    .sum()
)

print(
    f"Total Revenue At Risk: {total_revenue_at_risk:,.2f}"
)

Total Revenue At Risk: 1,379,824.99


In [28]:
# ==========================================
# REVENUE AT RISK BY SEGMENT
# ==========================================

risk_df.groupby(
    "PrioritySegment"
)["RevenueRiskScore"].sum()

C:\Users\Akshit\AppData\Local\Temp\ipykernel_10884\3570289515.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  risk_df.groupby(


PrioritySegment
Low       182514.244236
Medium    367726.659270
High      829584.086598
Name: RevenueRiskScore, dtype: float64

In [29]:
# ==========================================
# TOP 20 CUSTOMERS TO SAVE
# ==========================================

risk_df[
    [
        "CustomerID",
        "MonthlyRevenue_Clean",
        "ChurnProbability",
        "RevenueRiskScore",
        "PrioritySegment"
    ]
].head(20)

,CustomerID,MonthlyRevenue_Clean,ChurnProbability,RevenueRiskScore,PrioritySegment
24038,3189262,1223.38,0.563428,689.286820,High
7618,3059858,855.50,0.645068,551.855308,High
38290,3305434,861.11,0.627066,539.972869,High
31127,3247882,758.17,0.669646,507.705572,High
24874,3196158,575.75,0.773669,445.440135,High
23481,3184774,926.08,0.478221,442.870572,High
37456,3298662,523.20,0.785112,410.770486,High
37663,3300310,672.01,0.581583,390.829527,High
47654,3375582,847.82,0.451721,382.978008,High
24711,3194730,576.86,0.598706,345.369375,High


In [30]:
risk_df["PrioritySegment"].value_counts()

PrioritySegment
Low       17016
High      17016
Medium    17015
Name: count, dtype: int64

In [31]:
print(
    f"Total Revenue At Risk: {risk_df['RevenueRiskScore'].sum():,.2f}"
)

Total Revenue At Risk: 1,379,824.99
